# 02 - Baseline Models & Scientific Comparison

**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Notebook 02:** Random Forest & XGBoost Baseline Training, Evaluation, and Comparison  

This notebook evaluates reproducible baseline classifiers on the untouched 20% holdout test partition, extracts feature importances, and compares error profiles.


## 1. Setup & Environment


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import get_train_test_split
from src.train_rf import train_random_forest
from src.train_xgb import train_xgboost
from src.evaluation import evaluate_and_compare_models
from src.utils import get_baseline_results_dir, get_figures_dir, get_reports_dir, load_json


## 2. Train/Test Stratified Partitioning
Load strictly isolated 80% training ($n = 1400$) and 20% testing ($n = 350$) splits.


In [ ]:
X_train, X_test, y_train, y_test, feature_list = get_train_test_split()
print(f"Training shape: {X_train.shape} | Testing shape: {X_test.shape}")


## 3. Random Forest Baseline Training & Evaluation
Train `RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')` strictly on training data.


In [ ]:
rf_model, rf_metrics, rf_fi = train_random_forest()
print("Random Forest Metrics Summary:")
for k, v in rf_metrics.items():
    if isinstance(v, (int, float)):
        print(f"  {k:20s}: {v}")


## 4. XGBoost Baseline Training & Evaluation
Train `XGBClassifier(n_estimators=200, max_depth=6, lr=0.1, random_state=42)` on the identical training set.


In [ ]:
xgb_model, xgb_metrics, xgb_fi = train_xgboost()
print("XGBoost Metrics Summary:")
for k, v in xgb_metrics.items():
    if isinstance(v, (int, float)):
        print(f"  {k:20s}: {v}")


## 5. Baseline Model Comparison
Compare Accuracy, Balanced Accuracy, Precision, Recall, Macro F1, and Weighted F1 across both models.


In [ ]:
comp_df, comp_analysis = evaluate_and_compare_models()
display(comp_df)


## 6. Confusion Matrices
Inspect confusion matrices for Random Forest and XGBoost.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

classes = ["E", "G", "L"]
sns.heatmap(rf_metrics["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title("Random Forest Confusion Matrix", fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(xgb_metrics["confusion_matrix"], annot=True, fmt="d", cmap="Greens",
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title("XGBoost Confusion Matrix", fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()


## 7. Feature Importance Comparison (Top 20)


In [ ]:
plt.figure(figsize=(10, 6))
top_rf = rf_fi.head(15).sort_values("importance", ascending=True)
plt.barh(top_rf["feature"], top_rf["importance"], color="#1f77b4", edgecolor="black", alpha=0.85)
plt.title("Random Forest - Top 15 Feature Importances", fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
top_xgb = xgb_fi.head(15).sort_values("importance", ascending=True)
plt.barh(top_xgb["feature"], top_xgb["importance"], color="#2ca02c", edgecolor="black", alpha=0.85)
plt.title("XGBoost - Top 15 Feature Importances", fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()


## 8. Critical Research Synthesis & Discussion
> **Note on 100% Performance:**  
> Perfect performance requires further validation and should not automatically be interpreted as real-world ransomware detection capability.  
> As detailed in `results/reports/validation_summary.md`, both models discriminate between known families using cumulative post-execution sandbox totals. Phase 2 of this project transitions to sequential early-stage intervention via Proximal Policy Optimization (PPO).
